# Redrob candidate-ranking sandbox
Run all cells to reproduce the CPU-only ranking pipeline. The default input is the organizer-provided `sample_candidates.json` containing the first 50 anonymized candidates. Set `USE_ORGANIZER_SAMPLE = False` to upload a JSON, JSONL, or gzip candidate sample of your own.

The ranker gives the most weight to production career evidence in retrieval, ranking, evaluation, deployment, and ownership. It uses no GPU, model download, external API, or network call during ranking. Network access in the first cell is used only to fetch the three required repository files.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

workspace = Path('/content/redrob_demo')
workspace.mkdir(parents=True, exist_ok=True)
os.chdir(workspace)
sys.path.insert(0, str(workspace))
raw_base = 'https://raw.githubusercontent.com/Man1ac-1773/RedrobAI_hackathon/master'
required_files = ['rank.py', 'ranking_core.py', 'sample_candidates.json']
for filename in required_files:
    subprocess.run([
        'curl', '--fail', '--silent', '--show-error', '--location',
        '--output', filename, f'{raw_base}/{filename}',
    ], check=True)
print('Downloaded:', ', '.join(required_files))

In [ ]:
USE_ORGANIZER_SAMPLE = True

if USE_ORGANIZER_SAMPLE:
    candidate_path = workspace / 'sample_candidates.json'
else:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one candidate file')
    candidate_path = workspace / next(iter(uploaded))
print('Candidate input:', candidate_path)

In [ ]:
output_path = workspace / 'sandbox_ranking.csv'
subprocess.run([
    sys.executable, 'rank.py',
    '--candidates', str(candidate_path),
    '--out', str(output_path),
], check=True)

In [ ]:
import csv
from ranking_core import OUTPUT_COLUMNS, iter_candidates

source_ids = {candidate['candidate_id'] for candidate in iter_candidates(candidate_path)}
with output_path.open(encoding='utf-8', newline='') as handle:
    reader = csv.DictReader(handle)
    assert tuple(reader.fieldnames or ()) == OUTPUT_COLUMNS
    rows = list(reader)
assert len(rows) == min(100, len(source_ids))
assert [int(row['rank']) for row in rows] == list(range(1, len(rows) + 1))
assert len({row['candidate_id'] for row in rows}) == len(rows)
assert {row['candidate_id'] for row in rows} <= source_ids
assert all(float(rows[i]['score']) >= float(rows[i + 1]['score']) for i in range(len(rows) - 1))
assert all(row['reasoning'].strip() for row in rows)
print(f'PASS: generated and validated {len(rows)} ranked rows from {len(source_ids)} candidates')
for row in rows[:10]:
    print(f"#{row['rank']} {row['candidate_id']} score={row['score']}")
    print(row['reasoning'])

In [ ]:
DOWNLOAD_CSV = True
if DOWNLOAD_CSV:
    from google.colab import files
    files.download(str(output_path))